In [ ]:
import torch
from datasets import Dataset
from transformers import AutoTokenizer,AutoModelForSequenceClassification,TrainingArguments,Trainer,set_seed

set_seed(42)

data={
    "text":[
        "The transformer model achieved excellent accuracy.",
        "Large Language Models are revolutionizing AI.",
        "The football team won the championship.",
        "The cricket match was exciting.",
        "Neural networks are widely used in deep learning.",
        "The player scored a brilliant goal.",
        "Machine learning improves decision making.",
        "The tennis tournament starts tomorrow."
    ],
    "label":[1,1,0,0,1,0,1,0]
}

dataset=Dataset.from_dict(data)
print("Dataset:")
print(dataset)

model_name="bert-base-uncased"
tokenizer=AutoTokenizer.from_pretrained(model_name)

def tokenize_data(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

tokenized_dataset=dataset.map(
    tokenize_data,
    batched=True,
    remove_columns=["text"]
)
tokenized_dataset.reset_format()

print("\nTokenized Dataset:")
print(tokenized_dataset)

id2label={0:"Sports",1:"Technology"}
label2id={"Sports":0,"Technology":1}

model=AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    id2label=id2label,
    label2id=label2id
)

training_args=TrainingArguments(
    output_dir="./fine_tuned_model",
    per_device_train_batch_size=2,
    num_train_epochs=2,
    learning_rate=5e-5,
    logging_steps=1,
    save_strategy="no",
    report_to="none",
    seed=42
)

trainer=Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
)

print("\nStarting model training...")
training_result=trainer.train()
print("\nTraining completed successfully.")
print("Training Loss:",training_result.training_loss)

save_directory="./fine_tuned_model"
trainer.save_model(save_directory)
tokenizer.save_pretrained(save_directory)
print("\nModel saved successfully.")

prediction_tokenizer=AutoTokenizer.from_pretrained(save_directory)
prediction_model=AutoModelForSequenceClassification.from_pretrained(save_directory)

device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
prediction_model=prediction_model.to(device)
prediction_model.eval()

print("Using device:",device)

text="Generative AI models improve intelligent automation."

inputs=prediction_tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    padding=True,
    max_length=128
).to(device)

with torch.no_grad():
    outputs=prediction_model(**inputs)

probabilities=torch.softmax(outputs.logits,dim=-1)
predicted_id=torch.argmax(probabilities,dim=-1).item()
confidence_score=probabilities[0][predicted_id].item()
predicted_class=prediction_model.config.id2label[predicted_id]

print("\nPrediction")
print("-"*45)
print("Input:",text)
print("Predicted Class:",predicted_class)
print("Confidence Score:",round(confidence_score,3))

Dataset:
Dataset({
    features: ['text', 'label'],
    num_rows: 8
})


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

...


Training completed successfully.
Training Loss: 0.6021197959780693


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Model saved successfully.


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Using device: cuda

Prediction
---------------------------------------------
Input: Generative AI models improve intelligent automation.
Predicted Class: Technology
Confidence Score: 0.642
